Installing any needed libraries.


In [1]:
!pip install scipy statsmodels


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm


Loading dataset and reading

In [3]:
# Load the dataset
df = pd.read_csv('marketing_AB.csv')
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


Full data inventory

In [4]:
print("shape",df.shape)         # rows & columns
print("\ncolumn types\n",df.dtypes)        # column types
print("\nmissing values\n",df.isnull().sum()) # missing values
print("\ndata description\n",df.describe())

shape (588101, 7)

column types
 Unnamed: 0        int64
user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object

missing values
 Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

data description
           Unnamed: 0       user id      total ads  most ads hour
count  588101.000000  5.881010e+05  588101.000000  588101.000000
mean   294050.000000  1.310692e+06      24.820876      14.469061
std    169770.279667  2.022260e+05      43.715181       4.834634
min         0.000000  9.000000e+05       1.000000       0.000000
25%    147025.000000  1.143190e+06       4.000000      11.000000
50%    294050.000000  1.313725e+06      13.000000      14.000000
75%    441075.000000  1.484088e+06      27.000000      18.000000
max    588100.000000  1.654483e+06    2065.000000      23.000000


Group validation (96/4 split)

In [5]:
# Group validation — check the ad vs PSA split
print(df['test group'].value_counts())
print(df['test group'].value_counts(normalize=True) * 100)
# Expect ~96% ad, ~4% PSA

test group
ad     564577
psa     23524
Name: count, dtype: int64
test group
ad     96.000007
psa     3.999993
Name: proportion, dtype: float64


Rename columns, validate data, create "converted_int"

In [6]:
# Renaming columns to be cleaner (snake_case)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns)

# Validating the 'converted' column
print(df['converted'].value_counts())
print(df['converted'].dtype)

# Creating converted_int (True/False → 1/0)
df['converted_int'] = df['converted'].astype(int)
df[['converted', 'converted_int']].head()

Index(['unnamed:_0', 'user_id', 'test_group', 'converted', 'total_ads',
       'most_ads_day', 'most_ads_hour'],
      dtype='object')
converted
False    573258
True      14843
Name: count, dtype: int64
bool


,converted,converted_int
0,False,0
1,False,0
2,False,0
3,False,0
4,False,0


Create ads_bucket, is_weekend, hour_period features

In [7]:
# is_weekend
weekend_days = ['Saturday', 'Sunday']
df['is_weekend'] = df['most_ads_day'].isin(weekend_days).astype(int)

# hour_period
def hour_to_period(h):
    if 5 <= h < 12:
        return 'morning'
    elif 12 <= h < 17:
        return 'afternoon'
    elif 17 <= h < 21:
        return 'evening'
    else:
        return 'night'

df['hour_period'] = df['most_ads_hour'].apply(hour_to_period)

# ads_bucket — FIXED: 'total_ads' instead of 'total ads'
df['ads_bucket'] = pd.qcut(df['total_ads'], q=4, labels=['low', 'medium', 'high', 'very_high'])

# Verify
df[['most_ads_day', 'is_weekend', 'most_ads_hour', 'hour_period', 'total_ads', 'ads_bucket']].head(10)

,most_ads_day,is_weekend,most_ads_hour,hour_period,total_ads,ads_bucket
0,Monday,0,20,evening,130,very_high
1,Tuesday,0,22,night,93,very_high
2,Tuesday,0,18,evening,21,high
3,Tuesday,0,10,morning,355,very_high
4,Friday,0,14,afternoon,276,very_high
5,Saturday,1,10,morning,734,very_high
6,Wednesday,0,13,afternoon,264,very_high
7,Sunday,1,18,evening,17,high
8,Tuesday,0,19,evening,21,high
9,Monday,0,14,afternoon,142,very_high


In [8]:
# Saving the engineered dataframe so we don't have to redo this
df.to_csv('marketing_AB_engineered.csv', index=False)
print("Saved! Shape:", df.shape)

Saved! Shape: (588101, 11)


In [9]:
#downloading the cleaned data set
from google.colab import files
files.download('marketing_AB_engineered.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>